In [ ]:
#Environment Check
import sys
import cv2
import mediapipe as mp
import insightface
import ultralytics

print("Python:", sys.version)
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("InsightFace imported OK")
print("Ultralytics:", ultralytics.__version__)


Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\SUJAL\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Python: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
OpenCV: 4.12.0
MediaPipe: 0.10.14
InsightFace imported OK
Ultralytics: 8.3.231


In [ ]:
# Base Imports + Helper Class
import cv2
import mediapipe as mp
from dataclasses import dataclass

mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

@dataclass
class BBox:
    xmin: int
    ymin: int
    width: int
    height: int


In [ ]:
# Webcam Test
import cv2

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Cannot open webcam")

ret, frame = cap.read()
print("Frame captured:", ret, "Shape:", frame.shape if ret else None)

cap.release()


Frame captured: True Shape: (480, 640, 3)


In [5]:
# Real-Time Face Detection (MediaPipe)
import cv2
import time
import mediapipe as mp

mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Cannot open webcam")

with mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5) as face_detector:
    prev = time.time()
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Empty frame")
            continue
        
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_detector.process(rgb)

        if results.detections:
            for det in results.detections:
                mp_drawing.draw_detection(frame, det)

        now = time.time()
        fps = 1.0 / (now - prev)
        prev = now
        cv2.putText(frame, f"FPS: {fps:.1f}", (10,30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

        cv2.imshow("MediaPipe Face Detection", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


d:\sujal\dev\Machine learning projects\Face recog\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [17]:
# Capture Face ROI & Save to Folder
import cv2
import mediapipe as mp
import os
import time

os.makedirs("../data/registered_user", exist_ok=True)
mp_face_detection = mp.solutions.face_detection

cap = cv2.VideoCapture(0)

print("Webcam warming up...")
time.sleep(1)

start_time = time.time()
print("Showing live feed for 5 seconds...")

# Show video for 5 seconds
while time.time() - start_time < 5:
    ret, frame = cap.read()
    if not ret:
        continue
    cv2.imshow("Camera - Relax & Position Yourself", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

# Show countdown 3 seconds
for i in range(3, 0, -1):
    ret, frame = cap.read()
    if not ret:
        continue
    txt = f"Capturing in {i}..."
    cv2.putText(frame, txt, (50, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 3)
    cv2.imshow("Camera", frame)
    cv2.waitKey(1000)

# Capture real frame
ret, frame = cap.read()
if not ret:
    raise RuntimeError("Could not capture frame")

rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

with mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5) as fd:
    results = fd.process(rgb)

    if results.detections:
        print("Face detected, saving ROI...")
        
        det = results.detections[0]
        bbox = det.location_data.relative_bounding_box
        ih, iw, _ = frame.shape

        x = int(bbox.xmin * iw)
        y = int(bbox.ymin * ih)
        w = int(bbox.width * iw)
        h = int(bbox.height * ih)

        # Expand bounding box by 30%
        pad = int(0.3 * h)

        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(iw, x + w + pad)
        y2 = min(ih, y + h + pad)

        face = frame[y1:y2, x1:x2]

        # Resize to 256x256 to help InsightFace
        face = cv2.resize(face, (256, 256))

        fname = "../data/registered_user/face_0.png"
        cv2.imwrite(fname, face)
        print("Saved:", fname)

    else:
        print("No face detected. Try again.")

cap.release()
cv2.destroyAllWindows()
print("Done.")


Webcam warming up...
Showing live feed for 5 seconds...
Face detected, saving ROI...
Saved: ../data/registered_user/face_0.png
Done.


### Next Steps

1. Compute InsightFace embeddings for saved face images  
2. Implement gaze tracking using MediaPipe Iris  
3. Test YOLOv8 object detection  
4. Multi-face detection  
5. Combine all modules into a real-time proctor pipeline  



### 🎯 So the next set of notebook cells will be:
Cell 7 — Load InsightFace model

Load the ONNX model and prepare it.

Cell 8 — Load registered user images + compute embeddings

Convert faces to normalized format and generate embeddings.

Cell 9 — Save embeddings

Store them in .npy so the real-time system can load them quickly.

Cell 10 — Test Recognition (compare webcam face vs saved embedding)

In [18]:
# import insightface
from insightface.app import FaceAnalysis

# Initialize InsightFace model
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))

print("InsightFace model loaded.")


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\SUJAL/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\SUJAL/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\SUJAL/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\SUJAL/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\SUJAL/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.

In [19]:
#Generate Embeddings for Registered User
import cv2
import numpy as np
import os

registered_path = "../data/registered_user/"
embedding_list = []

files = [f for f in os.listdir(registered_path) if f.endswith(".png")]

for file in files:
    img = cv2.imread(os.path.join(registered_path, file))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    faces = app.get(img_rgb)

    if len(faces) == 0:
        print(f"No face found in {file}, skipping.")
        continue

    emb = faces[0].embedding  # 512-D vector
    embedding_list.append(emb)
    print(f"Embedding extracted from {file}")

embedding_list = np.array(embedding_list)
print("Total embeddings:", embedding_list.shape)


Embedding extracted from face_0.png
Total embeddings: (1, 512)


In [20]:
#Save User Embedding
os.makedirs("../data/embeddings/", exist_ok=True)

user_embedding = np.mean(embedding_list, axis=0)
np.save("../data/embeddings/user.npy", user_embedding)

print("Saved user embedding to ../data/embeddings/user.npy")


Saved user embedding to ../data/embeddings/user.npy


In [22]:
#Test Recognition With Webcam
import cv2
import numpy as np

saved_emb = np.load("../data/embeddings/user.npy")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

cap = cv2.VideoCapture(0)

print("Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        continue
    
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    faces = app.get(rgb)

    if len(faces) > 0:
        emb = faces[0].embedding
        sim = cosine_similarity(saved_emb, emb)

        text = f"Similarity: {sim:.3f}"
        cv2.putText(frame, text, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 
                    0.8, (0,255,0) if sim > 0.55 else (0,0,255), 2)

    cv2.imshow("Recognition Test", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



Press 'q' to quit.


## gaze tracking


Cell 11: Iris model load

Cell 12: Extract eye landmarks

Cell 13: Compute gaze direction

Cell 14: Real-time gaze tracker (with display)

In [23]:
import cv2
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,   # Needed for iris tracking
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

print("MediaPipe Iris model loaded.")


MediaPipe Iris model loaded.


In [24]:
import numpy as np

# MediaPipe Iris Landmark Indexes
LEFT_IRIS = [468, 469, 470, 471]
RIGHT_IRIS = [473, 474, 475, 476]

# Left eye outer/inner corners
LEFT_EYE = [33, 133]

# Right eye outer/inner corners
RIGHT_EYE = [362, 263]

def get_iris_position(landmarks, frame_w, frame_h, eye_indices):
    iris_pts = []
    for idx in eye_indices:
        x = int(landmarks[idx].x * frame_w)
        y = int(landmarks[idx].y * frame_h)
        iris_pts.append((x, y))
    return iris_pts


In [27]:
def compute_gaze(landmarks, w, h, eye_corner_idx, iris_idx, eye_lid_idx):
    # Horizontal gaze
    left_corner = landmarks[eye_corner_idx[0]]
    right_corner = landmarks[eye_corner_idx[1]]
    lc_x, rc_x = int(left_corner.x * w), int(right_corner.x * w)

    iris_pts = get_iris_position(landmarks, w, h, iris_idx)
    iris_x = int(np.mean([p[0] for p in iris_pts]))

    horizontal_ratio = (iris_x - lc_x) / (rc_x - lc_x)

    # Vertical gaze
    upper_lid = landmarks[eye_lid_idx[0]]
    lower_lid = landmarks[eye_lid_idx[1]]
    up_y, low_y = int(upper_lid.y * h), int(lower_lid.y * h)
    iris_y = int(np.mean([p[1] for p in iris_pts]))

    vertical_ratio = (iris_y - up_y) / (low_y - up_y)

    return horizontal_ratio, vertical_ratio


In [ ]:
# Upper and lower eyelid landmarks (MediaPipe indexes)
LEFT_EYE_LIDS = [159, 145]   # upper, lower
RIGHT_EYE_LIDS = [386, 374]  # upper, lower

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)

    if results.multi_face_landmarks:
        lm = results.multi_face_landmarks[0].landmark
        h, w, _ = frame.shape

        # compute both horizontal & vertical ratios
        horiz, vert = compute_gaze(
            lm, w, h,
            LEFT_EYE, LEFT_IRIS,
            LEFT_EYE_LIDS
        )

        # ↓↓↓ MORE RELAXED THRESHOLDS ↓↓↓
        if horiz < 0.25:
            gaze = "Left"
        elif horiz > 0.75:
            gaze = "Right"
        elif vert < 0.30:
            gaze = "Up"
        elif vert > 0.70:
            gaze = "Down"
        else:
            gaze = "Center"

        cv2.putText(frame, f"Gaze: {gaze}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                    (0, 255, 0), 2)

    cv2.imshow("Gaze Tracking (Improved)", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


## YOLOv8 Object Detection cells (phone, book, second person)


In [29]:

# Cell 15 — load YOLOv8 model
from ultralytics import YOLO

# Use yolov8n (small, fast). If you have a custom weights file, replace the string with the path.
model = YOLO('yolov8n.pt')

print("Loaded YOLO model. Model names/classes:", model.names)

Loaded YOLO model. Model names/classes: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: '

In [30]:
# Cell 16 — test YOLO on a single image and save annotated output
import os
import cv2
import numpy as np
from datetime import datetime

# path to test image (example uses registered face or change to any test image)
img_path = "../data/registered_user/face_0.png"
out_dir = "../data/yolo_results"
os.makedirs(out_dir, exist_ok=True)

# Run inference
results = model(img_path, imgsz=640, conf=0.25, iou=0.45)  # returns list-like, use first result
res = results[0]

# annotated image (numpy BGR)
annotated = res.plot()

# get detected class indices and names
det_classes = []
det_names = []
if hasattr(res, "boxes") and len(res.boxes) > 0:
    try:
        cls_arr = res.boxes.cls.cpu().numpy().astype(int)
    except Exception:
        # fallback if cpu() not required
        cls_arr = res.boxes.cls.numpy().astype(int)
    det_classes = [int(c) for c in cls_arr]
    det_names = [res.names[c] for c in det_classes]

print("Detections:", det_names)

# Save annotated image
out_path = os.path.join(out_dir, f"annotated_{datetime.now().strftime('%Y%m%d_%H%M%S')}.jpg")
cv2.imwrite(out_path, annotated)
print("Saved annotated image to", out_path)



image 1/1 d:\sujal\dev\Machine learning projects\Face recog\notebooks\..\data\registered_user\face_0.png: 640x640 (no detections), 110.8ms
Speed: 3.1ms preprocess, 110.8ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)
Detections: []
Saved annotated image to ../data/yolo_results\annotated_20251125_021109.jpg


In [31]:
# Cell 17 — real-time detection: show boxes and class names on webcam
import cv2
from datetime import datetime

target_classes = {"cell phone", "book", "person"}  # we will look for these names

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Cannot open webcam")

print("Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    # Run YOLO on the frame (streaming inference)
    results = model(frame, imgsz=640, conf=0.25, iou=0.45)
    res = results[0]

    # annotated frame (res.plot() returns a copy with boxes)
    annotated = res.plot()

    # count targets
    counts = {"person": 0, "cell phone": 0, "book": 0}
    if hasattr(res, "boxes") and len(res.boxes) > 0:
        try:
            cls_arr = res.boxes.cls.cpu().numpy().astype(int)
        except Exception:
            cls_arr = res.boxes.cls.numpy().astype(int)
        for c in cls_arr:
            name = res.names[int(c)]
            if name in counts:
                counts[name] += 1

    # overlay counts and simple warnings
    info = f"Person:{counts['person']}  Phone:{counts['cell phone']}  Book:{counts['book']}"
    cv2.putText(annotated, info, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    # Warning color when detected
    if counts['cell phone'] > 0:
        cv2.putText(annotated, "Phone detected!", (10,70), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)
    if counts['book'] > 0:
        cv2.putText(annotated, "Book detected!", (10,100), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)
    if counts['person'] > 1:
        cv2.putText(annotated, "ALERT: Multiple persons!", (10,130), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)

    cv2.imshow("YOLOv8 - Real-time Detection", annotated)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Press 'q' to quit.

0: 480x640 1 person, 84.4ms
Speed: 1.3ms preprocess, 84.4ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 95.6ms
Speed: 3.0ms preprocess, 95.6ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 96.8ms
Speed: 1.9ms preprocess, 96.8ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 tie, 111.2ms
Speed: 2.1ms preprocess, 111.2ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 tie, 88.0ms
Speed: 1.4ms preprocess, 88.0ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 82.3ms
Speed: 1.7ms preprocess, 82.3ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 90.5ms
Speed: 1.4ms preprocess, 90.5ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 104.6ms
Speed: 1.7ms preprocess, 104.6ms inference, 1.6ms pos

In [32]:
# Cell 18 — real-time detection + logging of suspicious events to CSV
import csv
import os
import time
from datetime import datetime

log_path = "../data/proctor_events.csv"
os.makedirs(os.path.dirname(log_path), exist_ok=True)

# create CSV header if not exists
if not os.path.exists(log_path):
    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "event_type", "details"])

# helper to log an event
def log_event(event_type, details=""):
    ts = datetime.now().isoformat(timespec="seconds")
    with open(log_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([ts, event_type, details])
    print(f"[{ts}] {event_type} - {details}")

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Cannot open webcam")

last_log_times = {"phone": 0, "book": 0, "multi_person": 0}
cooldown = 5  # seconds between repeated logs of the same type

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    results = model(frame, imgsz=640, conf=0.25, iou=0.45)
    res = results[0]

    # get counts
    counts = {"person": 0, "cell phone": 0, "book": 0}
    if hasattr(res, "boxes") and len(res.boxes) > 0:
        try:
            cls_arr = res.boxes.cls.cpu().numpy().astype(int)
        except Exception:
            cls_arr = res.boxes.cls.numpy().astype(int)
        for c in cls_arr:
            name = res.names[int(c)]
            if name in counts:
                counts[name] += 1

    # Logging logic with cooldowns
    now = time.time()
    if counts["cell phone"] > 0 and now - last_log_times["phone"] > cooldown:
        log_event("phone_detected", f"count={counts['cell phone']}")
        last_log_times["phone"] = now

    if counts["book"] > 0 and now - last_log_times["book"] > cooldown:
        log_event("book_detected", f"count={counts['book']}")
        last_log_times["book"] = now

    if counts["person"] > 1 and now - last_log_times["multi_person"] > cooldown:
        log_event("multi_person", f"count={counts['person']}")
        last_log_times["multi_person"] = now

    # Show annotated frame
    annotated = res.plot()
    info = f"P:{counts['person']}  Phone:{counts['cell phone']}  Book:{counts['book']}"
    cv2.putText(annotated, info, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
    cv2.imshow("YOLOv8 - Detection + Logging", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Event log saved to:", log_path)



0: 480x640 1 person, 1 wine glass, 78.7ms
Speed: 7.8ms preprocess, 78.7ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 96.2ms
Speed: 2.0ms preprocess, 96.2ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 wine glass, 99.9ms
Speed: 1.8ms preprocess, 99.9ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 wine glass, 109.5ms
Speed: 1.4ms preprocess, 109.5ms inference, 2.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 wine glass, 114.0ms
Speed: 1.9ms preprocess, 114.0ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 79.3ms
Speed: 1.3ms preprocess, 79.3ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 80.2ms
Speed: 1.5ms preprocess, 80.2ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 82.3ms
Speed: 1.3ms preprocess, 82.3

## multi face detection

In [33]:
import cv2
import mediapipe as mp

mp_face_detection = mp.solutions.face_detection
face_detector = mp_face_detection.FaceDetection(
    model_selection=0,
    min_detection_confidence=0.5
)

print("Multi-face detection ready.")


Multi-face detection ready.


In [34]:
# Function to Count Faces
def count_faces(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_detector.process(rgb)

    if not results.detections:
        return 0, []

    # get bounding boxes for visualization
    face_boxes = []
    h, w, _ = frame.shape
    for det in results.detections:
        rel = det.location_data.relative_bounding_box
        x = int(rel.xmin * w)
        y = int(rel.ymin * h)
        width = int(rel.width * w)
        height = int(rel.height * h)
        face_boxes.append((x, y, width, height))

    return len(face_boxes), face_boxes

In [35]:
# Real-Time Multi-Face Detection Loop
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Cannot open webcam")

print("Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    face_count, boxes = count_faces(frame)

    # Draw the face boxes
    for (x, y, w, h) in boxes:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)

    # Display counter
    cv2.putText(frame, f"Faces: {face_count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9,
                (0, 255, 0) if face_count <= 1 else (0, 0, 255), 2)

    # Warning if >1 face
    if face_count > 1:
        cv2.putText(frame, "ALERT: MULTIPLE FACES DETECTED!",
                    (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                    (0, 0, 255), 3)

    cv2.imshow("Multi-Face Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Press 'q' to quit.


## event logger

In [36]:
#Create the Log File + Logger Function
import csv
import os
from datetime import datetime

log_path = "../data/proctor_events.csv"
os.makedirs(os.path.dirname(log_path), exist_ok=True)

# Create header if file doesn't exist
if not os.path.exists(log_path):
    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "event", "details"])

def log_event(event, details=""):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(log_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([timestamp, event, details])
    print(f"[LOG] {timestamp} | {event} | {details}")

In [37]:
#Logging Rules (Cheating Detection Logic)

# This cell defines when to log events.

# Cooldowns (seconds) to avoid spamming logs
event_cooldown = {
    "face_mismatch": 5,
    "multi_face": 5,
    "look_away": 3,
    "phone_detected": 5,
    "book_detected": 5,
    "no_face": 3,
    "person_detected": 5
}

last_event_time = {k: 0 for k in event_cooldown}

import time

def should_log(event_name):
    now = time.time()
    if now - last_event_time[event_name] > event_cooldown[event_name]:
        last_event_time[event_name] = now
        return True
    return False

In [38]:
# Logger Helper for Each Type of Event


def log_face_mismatch(similarity):
    if should_log("face_mismatch"):
        log_event("FACE_MISMATCH", f"similarity={similarity:.3f}")

def log_multi_face(count):
    if should_log("multi_face"):
        log_event("MULTIPLE_FACES", f"faces={count}")

def log_look_away(direction):
    if should_log("look_away"):
        log_event("LOOK_AWAY", direction)

def log_phone_detected(count):
    if should_log("phone_detected"):
        log_event("PHONE_DETECTED", f"count={count}")

def log_book_detected(count):
    if should_log("book_detected"):
        log_event("BOOK_DETECTED", f"count={count}")

def log_no_face():
    if should_log("no_face"):
        log_event("NO_FACE_DETECTED", "User not visible")

def log_multi_person(count):
    if should_log("person_detected"):
        log_event("MULTIPLE_PERSON_DETECTED", f"count={count}")

In [39]:
# Quick Test (Manually Trigger Events)


log_event("TEST_START", "Proctor logger test initialized")

log_face_mismatch(0.31)
log_multi_face(2)
log_look_away("Right")
log_phone_detected(1)
log_book_detected(1)
log_no_face()
log_multi_person(2)

log_event("TEST_END", "Logger test complete")

[LOG] 2025-11-25 02:20:14 | TEST_START | Proctor logger test initialized
[LOG] 2025-11-25 02:20:14 | FACE_MISMATCH | similarity=0.310
[LOG] 2025-11-25 02:20:14 | MULTIPLE_FACES | faces=2
[LOG] 2025-11-25 02:20:14 | LOOK_AWAY | Right
[LOG] 2025-11-25 02:20:14 | PHONE_DETECTED | count=1
[LOG] 2025-11-25 02:20:14 | BOOK_DETECTED | count=1
[LOG] 2025-11-25 02:20:14 | NO_FACE_DETECTED | User not visible
[LOG] 2025-11-25 02:20:14 | MULTIPLE_PERSON_DETECTED | count=2
[LOG] 2025-11-25 02:20:14 | TEST_END | Logger test complete


## finally 

In [1]:
# Final integrated proctor loop
# Paste this whole cell into your notebook and run.
import cv2
import time
import numpy as np
import os
from datetime import datetime

# ----------------------------
# Config / thresholds
# ----------------------------
EMBED_PATH = "../data/embeddings/user.npy"
LOG_PATH = "../data/proctor_events.csv"
YOLO_IMG_SIZE = 640
YOLO_CONF = 0.25
RECOG_SIM_THRESHOLD = 0.55   # similarity threshold for match
NO_FACE_TIMEOUT = 4.0        # seconds considered "no face / left camera"
FRAME_SKIP_YOLO = 2          # run YOLO every N frames
FRAME_SKIP_RECOG = 1         # run recognition every N frames (1 => every frame)

# Gaze thresholds (relaxed)
HORIZ_LEFT = 0.25
HORIZ_RIGHT = 0.75
VERT_UP = 0.30
VERT_DOWN = 0.70

# Cooldowns for logging to avoid spam (seconds)
COOLDOWNS = {
    "face_mismatch": 5,
    "multi_face": 5,
    "look_away": 3,
    "phone_detected": 5,
    "book_detected": 5,
    "no_face": 3,
    "multi_person": 5
}
_last_event_time = {k: 0 for k in COOLDOWNS}

def should_log(key):
    now = time.time()
    if now - _last_event_time[key] > COOLDOWNS[key]:
        _last_event_time[key] = now
        return True
    return False

# ----------------------------
# Logger (use existing if available)
# ----------------------------
if "log_event" not in globals():
    os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)
    if not os.path.exists(LOG_PATH):
        with open(LOG_PATH, "w", newline="") as f:
            f.write("timestamp,event,details\n")
    def log_event(event, details=""):
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(LOG_PATH, "a", newline="") as f:
            f.write(f'{ts},{event},"{details}"\n')
        print(f"[LOG] {ts} | {event} | {details}")

# Convenience wrappers for common logs (if you prefer using them elsewhere)
def log_face_mismatch(sim):
    if should_log("face_mismatch"):
        log_event("FACE_MISMATCH", f"similarity={sim:.3f}")

def log_multi_face(count):
    if should_log("multi_face"):
        log_event("MULTIPLE_FACES", f"faces={count}")

def log_look_away(dir_str):
    if should_log("look_away"):
        log_event("LOOK_AWAY", dir_str)

def log_phone_detected(count):
    if should_log("phone_detected"):
        log_event("PHONE_DETECTED", f"count={count}")

def log_book_detected(count):
    if should_log("book_detected"):
        log_event("BOOK_DETECTED", f"count={count}")

def log_no_face():
    if should_log("no_face"):
        log_event("NO_FACE_DETECTED", "User not visible")

def log_multi_person(count):
    if should_log("multi_person"):
        log_event("MULTIPLE_PERSON_DETECTED", f"count={count}")

# ----------------------------
# Load user embedding
# ----------------------------
if not os.path.exists(EMBED_PATH):
    raise FileNotFoundError(f"User embedding not found at {EMBED_PATH}. Run embedding generation first.")
saved_emb = np.load(EMBED_PATH)

# ----------------------------
# Model initialization
# ----------------------------
# InsightFace
try:
    import insightface
    from insightface.app import FaceAnalysis
    app = FaceAnalysis(name="buffalo_l")
    app.prepare(ctx_id=0, det_size=(640,640))
except Exception as e:
    raise RuntimeError("Failed to load InsightFace (FaceAnalysis). Ensure insightface installed and C++ build tools present.") from e

# MediaPipe face detection + face mesh (for gaze)
import mediapipe as mp
mp_fd = mp.solutions.face_detection
mp_fm = mp.solutions.face_mesh

face_detector = mp_fd.FaceDetection(model_selection=0, min_detection_confidence=0.5)
face_mesh = mp_fm.FaceMesh(max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5, min_tracking_confidence=0.5)

# YOLOv8 (ultralytics)
from ultralytics import YOLO
model = YOLO('yolov8n.pt')  # change path if you have custom weights

# ----------------------------
# Gaze helpers (MediaPipe indexes)
# ----------------------------
LEFT_IRIS = [468, 469, 470, 471]
RIGHT_IRIS = [473, 474, 475, 476]
LEFT_EYE = [33, 133]
RIGHT_EYE = [362, 263]
LEFT_EYE_LIDS = [159, 145]   # upper, lower
RIGHT_EYE_LIDS = [386, 374]

def get_iris_position(landmarks, frame_w, frame_h, eye_indices):
    pts = []
    for idx in eye_indices:
        lm = landmarks[idx]
        pts.append((int(lm.x * frame_w), int(lm.y * frame_h)))
    return pts

def compute_gaze(landmarks, w, h, eye_corner_idx, iris_idx, eye_lid_idx):
    left_corner = landmarks[eye_corner_idx[0]]
    right_corner = landmarks[eye_corner_idx[1]]
    lc_x, rc_x = int(left_corner.x * w), int(right_corner.x * w)

    iris_pts = get_iris_position(landmarks, w, h, iris_idx)
    iris_x = int(np.mean([p[0] for p in iris_pts]))
    horiz = (iris_x - lc_x) / (rc_x - lc_x + 1e-6)

    upper_lid = landmarks[eye_lid_idx[0]]
    lower_lid = landmarks[eye_lid_idx[1]]
    up_y, low_y = int(upper_lid.y * h), int(lower_lid.y * h)
    iris_y = int(np.mean([p[1] for p in iris_pts]))
    vert = (iris_y - up_y) / (low_y - up_y + 1e-6)

    return horiz, vert

# ----------------------------
# Utility functions
# ----------------------------
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def expand_bbox(x, y, w, h, iw, ih, pad_ratio=0.3):
    pad = int(pad_ratio * h)
    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(iw, x + w + pad)
    y2 = min(ih, y + h + pad)
    return x1, y1, x2, y2

# ----------------------------
# Start capture loop
# ----------------------------
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Cannot open webcam")

print("Starting final proctor loop. Press 'q' to quit.")
last_face_time = time.time()
frame_idx = 0

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            continue
        frame_idx += 1
        h, w, _ = frame.shape
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # ----------------------------
        # Face detection (MediaPipe) -> multi-face counting + primary face bbox
        # ----------------------------
        fd_res = face_detector.process(frame_rgb)
        face_count = 0
        primary_face_box = None
        if fd_res.detections:
            face_count = len(fd_res.detections)
            # choose largest face as primary
            max_area = 0
            for det in fd_res.detections:
                rel = det.location_data.relative_bounding_box
                x = int(rel.xmin * w)
                y = int(rel.ymin * h)
                ww = int(rel.width * w)
                hh = int(rel.height * h)
                area = ww * hh
                if area > max_area:
                    max_area = area
                    primary_face_box = (x, y, ww, hh)
            last_face_time = time.time()
        else:
            # no face detected
            if time.time() - last_face_time > NO_FACE_TIMEOUT:
                log_no_face()

        # Draw face boxes
        if fd_res.detections:
            for det in fd_res.detections:
                rel = det.location_data.relative_bounding_box
                x = int(rel.xmin * w)
                y = int(rel.ymin * h)
                ww = int(rel.width * w)
                hh = int(rel.height * h)
                cv2.rectangle(frame, (x, y), (x+ww, y+hh), (50,205,50), 2)

        # Multi-face alert
        if face_count > 1:
            cv2.putText(frame, f"Faces: {face_count} (!) ", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)
            log_multi_face(face_count)
        else:
            cv2.putText(frame, f"Faces: {face_count}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

        # ----------------------------
        # Recognition (InsightFace) on primary face every FRAME_SKIP_RECOG frames
        # ----------------------------
        recognized_text = "Unknown"
        if primary_face_box is not None and (frame_idx % FRAME_SKIP_RECOG == 0):
            x, y, ww, hh = primary_face_box
            x1, y1, x2, y2 = expand_bbox(x, y, ww, hh, w, h, pad_ratio=0.3)
            face_crop = frame[y1:y2, x1:x2]
            # ensure crop not empty
            if face_crop.size != 0:
                try:
                    face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
                    faces_info = app.get(face_rgb)
                    if faces_info and len(faces_info) > 0:
                        emb = faces_info[0].embedding
                        sim = cosine_similarity(saved_emb, emb)
                        if sim >= RECOG_SIM_THRESHOLD:
                            recognized_text = f"User ({sim:.2f})"
                        else:
                            recognized_text = f"Mismatch ({sim:.2f})"
                            log_face_mismatch(sim)
                    else:
                        recognized_text = "NoFaceInCrop"
                except Exception as e:
                    recognized_text = "RecogErr"
            else:
                recognized_text = "EmptyCrop"

        cv2.putText(frame, f"ID: {recognized_text}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0), 2)

        # ----------------------------
        # Gaze tracking (face_mesh) - only if face present
        # ----------------------------
        gaze_state = "N/A"
        if primary_face_box is not None:
            mesh_res = face_mesh.process(frame_rgb)
            if mesh_res.multi_face_landmarks:
                lm = mesh_res.multi_face_landmarks[0].landmark
                # compute using left eye (could average both eyes)
                horiz, vert = compute_gaze(lm, w, h, LEFT_EYE, LEFT_IRIS, LEFT_EYE_LIDS)
                # determine discrete gaze
                if horiz < HORIZ_LEFT:
                    gaze_state = "Left"
                elif horiz > HORIZ_RIGHT:
                    gaze_state = "Right"
                elif vert < VERT_UP:
                    gaze_state = "Up"
                elif vert > VERT_DOWN:
                    gaze_state = "Down"
                else:
                    gaze_state = "Center"

                # only log on away events (Left/Right/Up/Down), not Center
                if gaze_state != "Center" and gaze_state != "N/A":
                    log_look_away(gaze_state)

        cv2.putText(frame, f"Gaze: {gaze_state}", (10, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (200,200,200), 2)

        # ----------------------------
        # YOLO object detection (run every FRAME_SKIP_YOLO frames)
        # ----------------------------
        phone_count = 0
        book_count = 0
        person_count_yolo = 0
        if frame_idx % FRAME_SKIP_YOLO == 0:
            try:
                yolo_res = model(frame, imgsz=YOLO_IMG_SIZE, conf=YOLO_CONF, iou=0.45)[0]
                # annotated copy
                annotated = yolo_res.plot()
                # classes array
                if hasattr(yolo_res, "boxes") and len(yolo_res.boxes) > 0:
                    try:
                        cls_arr = yolo_res.boxes.cls.cpu().numpy().astype(int)
                    except Exception:
                        cls_arr = yolo_res.boxes.cls.numpy().astype(int)
                    for c in cls_arr:
                        name = yolo_res.names[int(c)]
                        if name in ("cell phone", "mobile phone", "phone", "cellphone"):
                            phone_count += 1
                        if name in ("book",):
                            book_count += 1
                        if name == "person":
                            person_count_yolo += 1
                # update frame with annotated results
                frame = annotated
            except Exception as e:
                # YOLO inference failed this frame; continue silently
                pass

        # overlay counts and warnings
        info = f"PYOLO:{person_count_yolo} Phone:{phone_count} Book:{book_count}"
        cv2.putText(frame, info, (10, h-20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

        if phone_count > 0:
            cv2.putText(frame, "Phone detected!", (10, h-60), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)
            log_phone_detected(phone_count)
        if book_count > 0:
            cv2.putText(frame, "Book detected!", (10, h-95), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)
            log_book_detected(book_count)
        if person_count_yolo > 1:
            cv2.putText(frame, "ALERT: Extra person (YOLO)!", (10, h-130), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)
            log_multi_person(person_count_yolo)

        # ----------------------------
        # Show FPS
        # ----------------------------
        # basic FPS calc (smoothed)
       # ----------------------------
# Show FPS (fixed)
# ----------------------------
        if 'last_ts' not in globals():
         last_ts = time.time()
         fps_val = 0.0
 
        now = time.time()
        dt = now - last_ts
        if dt > 0:
          fps_val = 0.9 * fps_val + 0.1 * (1.0 / dt)

        last_ts = now

        cv2.putText(frame, f"FPS: {fps_val:.1f}", (w-140, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        cv2.imshow("AI Proctor - Integrated", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    # cleanup mediapipe contexts
    face_detector.close()
    face_mesh.close()
    print("Proctor loop ended.")


d:\sujal\dev\Machine learning projects\Face recog\venv\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\SUJAL/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\SUJAL/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\SUJAL/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\SUJAL/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\SUJAL/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.

d:\sujal\dev\Machine learning projects\Face recog\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


[LOG] 2025-11-25 19:15:36 | MULTIPLE_FACES | faces=2
[LOG] 2025-11-25 19:15:36 | FACE_MISMATCH | similarity=0.009
[LOG] 2025-11-25 19:15:37 | LOOK_AWAY | Up

0: 480x640 2 persons, 1 chair, 113.9ms
Speed: 12.9ms preprocess, 113.9ms inference, 2.7ms postprocess per image at shape (1, 3, 480, 640)
[LOG] 2025-11-25 19:15:37 | MULTIPLE_PERSON_DETECTED | count=2

0: 480x640 3 persons, 1 chair, 78.5ms
Speed: 1.6ms preprocess, 78.5ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 1 chair, 82.7ms
Speed: 1.3ms preprocess, 82.7ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 1 chair, 1 toothbrush, 79.6ms
Speed: 1.8ms preprocess, 79.6ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)
[LOG] 2025-11-25 19:15:40 | LOOK_AWAY | Up

0: 480x640 2 persons, 1 chair, 1 toothbrush, 79.1ms
Speed: 1.5ms preprocess, 79.1ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 tv